# Import necessary libraries

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import math
import uuid
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from functools import lru_cache
from math import radians, sin, cos, asin, sqrt
from pathlib import Path
import os
import re
import logging

from pandas.api.types import is_datetime64_any_dtype

# Dynamic Routing

## 1. Utility functions: distance, normalization, time window

In [ ]:

def _round5(x):
    """Round to 1e-5° (~1m) and turn NaN into None (hashable for lru_cache)."""
    return None if pd.isna(x) else round(float(x), 5)

@lru_cache(maxsize=2_000_000)
def _hav_cached(lon1, lat1, lon2, lat2):
    if None in (lon1, lat1, lon2, lat2):
        return 0.0
    R = 6371.0
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

def haversine_km(lon1, lat1, lon2, lat2):
    return _hav_cached(_round5(lon1), _round5(lat1), _round5(lon2), _round5(lat2))

def normalize_requests_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["pickup_datetime", "dropoff_datetime"]:
        if col in df.columns:
            if not is_datetime64_any_dtype(df[col]):
                df[col] = pd.to_datetime(
                    df[col],
                    errors="coerce",
                    format="mixed",
                )

            dtype = df[col].dtype
            if isinstance(dtype, pd.DatetimeTZDtype):
                df[col] = df[col].dt.tz_convert(None)

    if 'pickup_lon' not in df.columns and 'pickup_longitude' in df.columns:
        df['pickup_lon'] = df['pickup_longitude']
    if 'pickup_lat' not in df.columns and 'pickup_latitude' in df.columns:
        df['pickup_lat'] = df['pickup_latitude']
    if 'dropoff_lon' not in df.columns and 'dropoff_longitude' in df.columns:
        df['dropoff_lon'] = df['dropoff_longitude']
    if 'dropoff_lat' not in df.columns and 'dropoff_latitude' in df.columns:
        df['dropoff_lat'] = df['dropoff_latitude']

    if 'assigned' not in df.columns:
        df['assigned'] = False
    if 'assigned_taxi_id' not in df.columns:
        df['assigned_taxi_id'] = pd.Series([None] * len(df), dtype='object')
    if 'assigned_depot_id' not in df.columns:
        df['assigned_depot_id'] = pd.Series([None] * len(df), dtype='object')
    if 'assigned_mode' not in df.columns:
        df['assigned_mode'] = pd.Series([''] * len(df), dtype='object')

    return df

def ensure_pickup_windows(df: pd.DataFrame, early_s: int = 0, late_s: int = 180) -> pd.DataFrame:
    if 'pickup_earliest' not in df.columns:
        df['pickup_earliest'] = pd.to_datetime(df['pickup_datetime']) - pd.Timedelta(seconds=early_s)
    else:
        df['pickup_earliest'] = pd.to_datetime(df['pickup_earliest'])
    if 'pickup_latest' not in df.columns:
        df['pickup_latest']  = pd.to_datetime(df['pickup_datetime']) + pd.Timedelta(seconds=late_s)
    else:
        df['pickup_latest']  = pd.to_datetime(df['pickup_latest'])
    return df


def travel_minutes_km(km: float, speed_kmh: float = 25.0) -> float:
    return 0.0 if speed_kmh <= 0 else (km / speed_kmh) * 60.0

FARE_PER_HOUR_DRIVE   = 80.0
COST_PER_HOUR_DRIVE   = 5.0
COST_PER_HOUR_WAIT    = 1.0

def compute_arc_profit_paper(
    prev_lon: float,
    prev_lat: float,
    prev_end_time,
    req_row,
    speed_kmh: float,
    fare_per_hour: float = FARE_PER_HOUR_DRIVE,
    cost_drive_per_hour: float = COST_PER_HOUR_DRIVE,
    cost_wait_per_hour: float = COST_PER_HOUR_WAIT,
):
    """
    Calculate profit R_{i,c} for an arc: i -> c according to style paper.

    i = taxi (first arc) or previous customer (serial arc).

    Recipe:
      - T_dead (h) = distance(prev -> pickup) / speed_kmh
      - T_trip (h) = distance(pickup -> dropoff) / speed_kmh
      - arrival_time = prev_end_time + T_dead
      - wait_h = max(0, t_min_c - arrival_time)
      - Fare(c) = fare_per_hour * T_trip
      - Cost_drive = cost_drive_per_hour * (T_dead + T_trip)
      - Cost_wait  = cost_wait_per_hour  * wait_h
      - Profit = Fare - Cost_drive - Cost_wait
    """
    km_dead = haversine_km(prev_lon, prev_lat,
                           req_row["pickup_lon"], req_row["pickup_lat"])
    km_trip = haversine_km(req_row["pickup_lon"], req_row["pickup_lat"],
                           req_row["dropoff_lon"], req_row["dropoff_lat"])

    v = max(speed_kmh, 1e-9)
    t_dead_h = km_dead / v
    t_trip_h = km_trip / v

    arrival_time = prev_end_time + pd.to_timedelta(t_dead_h, unit="h")

    t_min_c = req_row["pickup_earliest"]
    wait_seconds = max(0.0, (t_min_c - arrival_time).total_seconds())
    wait_h = wait_seconds / 3600.0

    fare = fare_per_hour * t_trip_h
    cost_drive = cost_drive_per_hour * (t_dead_h + t_trip_h)
    cost_wait = cost_wait_per_hour * wait_h
    profit = fare - cost_drive - cost_wait

    return {
        "km_dead": km_dead,
        "km_trip": km_trip,
        "t_dead_h": t_dead_h,
        "t_trip_h": t_trip_h,
        "wait_h": wait_h,
        "fare_paper": fare,
        "cost_drive_paper": cost_drive,
        "cost_wait_paper": cost_wait,
        "profit_arc_paper": profit,
    }


## 2. Minimal static group bootstrap (necessary for dynamic simulation)

In [ ]:
@dataclass
class Taxi:
    taxi_id: str
    depot_id: str
    current_lon: float
    current_lat: float
    start_time: pd.Timestamp
    current_time: Optional[pd.Timestamp] = None

    def __post_init__(self):
        if self.current_time is None:
            self.current_time = self.start_time

@dataclass
class Fleet:
    taxis: List[Taxi]

    @classmethod
    def from_depots(cls, df: pd.DataFrame) -> "Fleet":
        if 'pickup_datetime' not in df.columns:
            raise ValueError("Missing 'pickup_datetime'.")
        start_time = pd.to_datetime(df['pickup_datetime']).min()
        if pd.isna(start_time):
            raise ValueError("Cannot determine start_time from pickup_datetime.")

        required = ['depot_id', 'depot_lon', 'depot_lat', 'taxis_per_depot']
        for c in required:
            if c not in df.columns:
                raise ValueError(f"Missing column '{c}' for Fleet construction.")

        depot_table = (
            df[required]
            .dropna(subset=['depot_id'])
            .drop_duplicates(subset=['depot_id'])
            .copy()
        )
        depot_table['taxis_per_depot'] = (
            pd.to_numeric(depot_table['taxis_per_depot'], errors='coerce')
              .fillna(0)
              .clip(lower=0)
              .round()
              .astype(int)
        )

        taxis: List[Taxi] = []
        for _, d in depot_table.iterrows():
            depot_id = str(d['depot_id'])
            depot_lon = float(d['depot_lon'])
            depot_lat = float(d['depot_lat'])
            n_taxis = int(d['taxis_per_depot'])
            for k in range(1, n_taxis + 1):
                taxis.append(Taxi(
                    taxi_id=f"{depot_id}-{k:03d}",
                    depot_id=depot_id,
                    current_lon=depot_lon,
                    current_lat=depot_lat,
                    start_time=start_time
                ))
        return cls(taxis=taxis)

## 3. TaxiLive status and build from Fleet

In [ ]:
@dataclass
class TaxiLive:
    taxi_id: str
    depot_id: str
    origin_lon: float
    origin_lat: float

    cur_lon: float
    cur_lat: float
    cur_time: pd.Timestamp

    route_freeze: List[int] = field(default_factory=list)
    route_open:   List[int] = field(default_factory=list)
    freeze_ptr: int = 0
    reposition_km: float = 0.0


def build_taxis_live(df_out: pd.DataFrame, fleet: Fleet) -> Dict[str, TaxiLive]:
    """Initialize TaxiLive objects; origin from df_out's depot_map (fallback: fleet/current)."""
    start = pd.to_datetime(
        df_out['pickup_earliest'] if 'pickup_earliest' in df_out.columns else df_out['pickup_datetime']
    ).min()

    depot_map: Dict[str, Tuple[float, float]] = {}
    if {'depot_id', 'depot_lon', 'depot_lat'}.issubset(df_out.columns):
        dep = (df_out[['depot_id','depot_lon','depot_lat']]
               .dropna(subset=['depot_id'])
               .drop_duplicates('depot_id'))
        for _, r in dep.iterrows():
            depot_map[str(r['depot_id'])] = (float(r['depot_lon']), float(r['depot_lat']))

    taxis: Dict[str, TaxiLive] = {}
    for t in getattr(fleet, 'taxis', []):
        dep_id = getattr(t, 'depot_id', None)
        dlon = dlat = None
        if dep_id is not None and str(dep_id) in depot_map:
            dlon, dlat = depot_map[str(dep_id)]
        else:
            dlon = getattr(t, 'depot_lon', None)
            dlat = getattr(t, 'depot_lat', None)
        if dlon is None or dlat is None:
            dlon = getattr(t, 'current_lon', 0.0)
            dlat = getattr(t, 'current_lat', 0.0)

        taxis[t.taxi_id] = TaxiLive(
            taxi_id=t.taxi_id,
            depot_id=str(dep_id) if dep_id is not None else "",
            origin_lon=float(dlon),
            origin_lat=float(dlat),
            cur_lon=float(dlon),
            cur_lat=float(dlat),
            cur_time=getattr(t, 'current_time', start)
        )
    return taxis

## 4. Feasibility and cost

In [ ]:

"""
It takes in a taxi state (tx) and a sequence of picks (freeze_seq + open_seq).
It simulates a taxi driving along this route, calculating the time of arrival (ETA) for each pick-up point.
If at any pickup point, ETA > pickup_latest (latest pickup time), the function returns False (not feasible).
It also returns total_wait_s (total early wait time) as a secondary criterion (tie-break).
"""
def route_feasible_from_state(df: pd.DataFrame, tx: TaxiLive, freeze_seq: List[int], open_seq: List[int],
                              speed_kmh: float = 25.0, return_lateness: bool = False):
    cur_t = tx.cur_time
    cur_lon, cur_lat = tx.cur_lon, tx.cur_lat
    total_wait_s = 0.0

    for ridx in (freeze_seq + open_seq):
        r = df.loc[ridx]
        km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        eta = cur_t + pd.to_timedelta(travel_minutes_km(km_dead, speed_kmh), unit='m')
        px  = r['pickup_datetime']
        px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else px
        px_l = r['pickup_latest']   if 'pickup_latest'   in df.columns else px

        if pd.notna(px_l) and eta > px_l:
            return (False, 0.0) if return_lateness else False
        if pd.notna(px_e):
            wait = (px_e - eta).total_seconds()
            if wait > 0:
                total_wait_s += wait

        cur_t = max(eta, px_e) if pd.notna(px_e) else eta

        km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_t = cur_t + pd.to_timedelta(travel_minutes_km(km_load, speed_kmh), unit='m')
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']

    return (True, total_wait_s) if return_lateness else True

def route_cost_km_current(df: pd.DataFrame, tx: TaxiLive, freeze_seq: List[int], open_seq: List[int]) -> float:
    cur_lon, cur_lat = tx.cur_lon, tx.cur_lat
    total = 0.0
    for ridx in (freeze_seq + open_seq):
        r = df.loc[ridx]
        total += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        total += haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return total

def route_cost_km_from_origin(df: pd.DataFrame, seq: List[int], start_lon: float, start_lat: float) -> float:
    cur_lon, cur_lat = start_lon, start_lat
    total = 0.0
    for ridx in seq:
        r = df.loc[ridx]
        total += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        total += haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return total

def deadhead_km_from_origin(df: pd.DataFrame, seq: List[int], start_lon: float, start_lat: float) -> float:
    cur_lon, cur_lat = start_lon, start_lat
    dead = 0.0
    for ridx in seq:
        r = df.loc[ridx]
        dead += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return dead

## 5. Status progression

In [ ]:

def update_freeze_open(df: pd.DataFrame, taxis: Dict[str, TaxiLive], now: pd.Timestamp, freeze_window_s: int):
    lock_until = now + pd.Timedelta(seconds=freeze_window_s)
    for tx in taxis.values():
        k = 0
        while k < len(tx.route_open):
            ridx = tx.route_open[k]
            px_e = df.at[ridx, 'pickup_earliest'] if 'pickup_earliest' in df.columns else df.at[ridx, 'pickup_datetime']
            if pd.isna(px_e) or px_e > lock_until:
                break
            k += 1
        if k > 0:
            tx.route_freeze.extend(tx.route_open[:k])
            tx.route_open = tx.route_open[k:]

def advance_state_after_freeze(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float):
    for tx in taxis.values():
        start_i = tx.freeze_ptr
        if start_i >= len(tx.route_freeze):
            continue

        cur_t  = tx.cur_time
        cur_lon, cur_lat = tx.cur_lon, tx.cur_lat

        for ridx in tx.route_freeze[start_i:]:
            r = df.loc[ridx]
            km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
            eta = cur_t + pd.to_timedelta(travel_minutes_km(km_dead, speed_kmh), unit='m')
            px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else r['pickup_datetime']

            cur_t = max(eta, px_e) if pd.notna(px_e) else eta
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_pickup_datetime'] = cur_t

            km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
            cur_t = cur_t + pd.to_timedelta(travel_minutes_km(km_load, speed_kmh), unit='m')
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_dropoff_datetime'] = cur_t

            cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']

        tx.cur_time, tx.cur_lon, tx.cur_lat = cur_t, cur_lon, cur_lat
        tx.freeze_ptr = len(tx.route_freeze)

def materialize_sim_times(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float):
    sim_dtype = "datetime64[ns]"
    if "sim_pickup_datetime" not in df.columns:
        df["sim_pickup_datetime"] = pd.Series(pd.NaT, index=df.index, dtype=sim_dtype)
    else:
        df["sim_pickup_datetime"] = pd.to_datetime(df["sim_pickup_datetime"], errors="coerce")
        try:
            df["sim_pickup_datetime"] = df["sim_pickup_datetime"].dt.tz_localize(None)
        except TypeError:
            pass

    if "sim_dropoff_datetime" not in df.columns:
        df["sim_dropoff_datetime"] = pd.Series(pd.NaT, index=df.index, dtype=sim_dtype)
    else:
        df["sim_dropoff_datetime"] = pd.to_datetime(df["sim_dropoff_datetime"], errors="coerce")
        try:
            df["sim_dropoff_datetime"] = df["sim_dropoff_datetime"].dt.tz_localize(None)
        except TypeError:
            pass

    to_minutes = lambda km: pd.to_timedelta(travel_minutes_km(km, speed_kmh), unit='m')

    for tx in taxis.values():
        seq = tx.route_open
        cur_t  = tx.cur_time
        cur_lon, cur_lat = tx.cur_lon, tx.cur_lat

        for ridx in seq:
            r = df.loc[ridx]
            km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
            eta = cur_t + to_minutes(km_dead)
            px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else r['pickup_datetime']

            cur_t = max(eta, px_e) if pd.notna(px_e) else eta
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_pickup_datetime'] = cur_t

            km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
            cur_t = cur_t + to_minutes(km_load)

            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)
            df.at[ridx, 'sim_dropoff_datetime'] = cur_t

            cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']


def compute_profit_paper_style(
    df: pd.DataFrame,
    taxis_live: dict,
    speed_kmh: float,
    fare_per_hour: float = FARE_PER_HOUR_DRIVE,
    cost_drive_per_hour: float = COST_PER_HOUR_DRIVE,
    cost_wait_per_hour: float = COST_PER_HOUR_WAIT,
):
    """
    Post-processing after running dynamic routing:
    - For each taxi, browse the requests it serves in turn (according to sim_pickup_datetime),
      and calculate R_{i,c} for each arc:
        * first arc: taxi k (at depot) -> first request c
        * next arc: previous request -> current request
    - Write the fare, cost, profit values ​​into df (each line = 1 request)
    - Returns:
        df has added columns,
        profit_by_taxi (dict),
        system_profit (total profit of the entire system)
    """
    df = df.copy()

    for col in ["fare_paper",
                "cost_drive_paper",
                "cost_wait_paper",
                "profit_arc_paper",
                "paper_arc_index",
                "km_dead_paper",
                "km_trip_paper",
                "km_total_paper"]:
        if col not in df.columns:
            df[col] = 0.0

    mask = (df.get("assigned", True) == True) & df["sim_pickup_datetime"].notna()
    served = df[mask]

    profit_by_taxi = {}
    system_profit = 0.0

    for tid, grp in served.groupby("assigned_taxi_id"):
        if pd.isna(tid):
            continue

        grp = grp.sort_values("sim_pickup_datetime").copy()

        tx = taxis_live[str(tid)] if isinstance(taxis_live, dict) else taxis_live[tid]

        prev_lon = getattr(tx, "origin_lon", getattr(tx, "cur_lon", 0.0))
        prev_lat = getattr(tx, "origin_lat", getattr(tx, "cur_lat", 0.0))

        start_time = pd.to_datetime(grp["pickup_earliest"]).min()
        last_dropoff_time = start_time

        total_fare = 0.0
        total_cost_drive = 0.0
        total_cost_wait = 0.0
        total_profit = 0.0

        arc_idx = 0

        for ridx, r in grp.iterrows():
            prev_time = last_dropoff_time

            res = compute_arc_profit_paper(
                prev_lon=prev_lon,
                prev_lat=prev_lat,
                prev_end_time=prev_time,
                req_row=r,
                speed_kmh=speed_kmh,
                fare_per_hour=fare_per_hour,
                cost_drive_per_hour=cost_drive_per_hour,
                cost_wait_per_hour=cost_wait_per_hour,
            )

            df.at[ridx, "fare_paper"] = res["fare_paper"]
            df.at[ridx, "cost_drive_paper"] = res["cost_drive_paper"]
            df.at[ridx, "cost_wait_paper"] = res["cost_wait_paper"]
            df.at[ridx, "profit_arc_paper"] = res["profit_arc_paper"]
            df.at[ridx, "paper_arc_index"] = float(arc_idx)

            df.at[ridx, "km_dead_paper"] = res["km_dead"]
            df.at[ridx, "km_trip_paper"] = res["km_trip"]
            df.at[ridx, "km_total_paper"] = res["km_dead"] + res["km_trip"]

            prev_lon = r["dropoff_lon"]
            prev_lat = r["dropoff_lat"]
            last_dropoff_time = r["sim_dropoff_datetime"]
            arc_idx += 1

            total_fare       += res["fare_paper"]
            total_cost_drive += res["cost_drive_paper"]
            total_cost_wait  += res["cost_wait_paper"]
            total_profit     += res["profit_arc_paper"]

        profit_by_taxi[str(tid)] = {
            "total_fare_paper": total_fare,
            "total_cost_drive_paper": total_cost_drive,
            "total_cost_wait_paper": total_cost_wait,
            "total_profit_paper": total_profit,
        }
        system_profit += total_profit

    return df, profit_by_taxi, system_profit


def print_top5_taxis_profit_from_df(
    df: pd.DataFrame,
    taxis_live: Dict[str, TaxiLive],
    speed_kmh: float,
    top_n: int = 5,
):
    """
    Calculate profit according to the paper formula and print out the top `top_n` taxis
    The biggest PROFIT and the biggest LOSS of the day.

    - df: dataframe after running the simulation (df2)
    - taxis_live: dict taxi_id -> TaxiLive (taxis2)
    - speed_kmh: model speed (for example 25.0)
    """
    df_profit, profit_by_taxi, system_profit = compute_profit_paper_style(
        df,
        taxis_live,
        speed_kmh=speed_kmh,
    )

    print(f"\nTotal system profit (paper-style): {system_profit:.2f}")

    rows = []
    for tid, stats in profit_by_taxi.items():
        rows.append({
            "taxi_id": tid,
            "total_profit": float(stats.get("total_profit_paper", 0.0)),
            "total_fare": float(stats.get("total_fare_paper", 0.0)),
            "total_cost_drive": float(stats.get("total_cost_drive_paper", 0.0)),
            "total_cost_wait": float(stats.get("total_cost_wait_paper", 0.0)),
        })

    dfp = pd.DataFrame(rows)
    if dfp.empty:
        print("No taxis have profit (profit_by_taxi is empty).")
        return

    dfp_sorted = dfp.sort_values("total_profit", ascending=False)
    top_n = min(top_n, len(dfp_sorted))

    print(f"\n=== Top {top_n} taxis with the most LOTS ===")
    print(dfp_sorted.head(top_n).to_string(index=False))

    print(f"\n=== Top {top_n} taxis with the most LOSS ===")
    print(
        dfp_sorted.tail(top_n)
        .sort_values("total_profit")
        .to_string(index=False)
    )

## 6. Candidate evaluation and insertion functions

In [ ]:

def _eta_minutes_to_point(tx: TaxiLive, lon: float, lat: float, speed_kmh: float) -> float:
    km = haversine_km(tx.cur_lon, tx.cur_lat, lon, lat)
    return travel_minutes_km(km, speed_kmh)

def nearest_taxis_for_point_eta(taxis: Dict[str, TaxiLive], lon: float, lat: float, k: int, speed_kmh: float) -> List[TaxiLive]:
    lst = list(taxis.values())
    lst.sort(key=lambda tx: _eta_minutes_to_point(tx, lon, lat, speed_kmh))
    return lst[:max(1, min(k, len(lst)))]

"""
    Find the nearest K taxi.
    For each taxi, try inserting request (ridx) into the first P positions and the last P positions of route_open.
    For each test insert, call route_feasible_from_state to test.
    Calculate delta = new_cost - base_cost.
    Find the (taxi, location) for the best non-negative delta (favor negative delta, then small positive delta, then lowest wait_s).
"""
def try_insert_best(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], ridx: int, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, pos_limit: int = 6
) -> bool:
    ALLOW_SMALL_POS_DELTA = False
    POS_DELTA_CAP_KM = 0.3
    POS_DELTA_FRAC_OF_TRIP = 0.15

    row = df.loc[ridx]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']

    def _ok_small_positive(delta: float) -> bool:
        if not ALLOW_SMALL_POS_DELTA:
            return False
        trip_km = haversine_km(row['pickup_lon'], row['pickup_lat'],
                               row['dropoff_lon'], row['dropoff_lat'])
        cap = min(float(POS_DELTA_CAP_KM), float(POS_DELTA_FRAC_OF_TRIP) * max(trip_km, 1e-9))
        return delta <= cap + 1e-9

    def _attempt(K, P) -> Optional[Tuple[float, str, int, float]]:
        cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, K, speed_kmh)
        best: Optional[Tuple[float, str, int, float]] = None
        for tx in cand_taxis:
            m = len(tx.route_open)
            positions = (sorted(set(list(range(0, min(P, m) + 1)) + list(range(max(0, m - P), m + 1))))
                         if P < m else list(range(0, m+1)))
            for pos in positions:
                cand_open = tx.route_open[:pos] + [ridx] + tx.route_open[pos:]
                ok, wait_s = route_feasible_from_state(df, tx, [], cand_open, speed_kmh, return_lateness=True)
                if not ok:
                    continue
                base = route_cost_km_current(df, tx, [], tx.route_open)
                newc = route_cost_km_current(df, tx, [], cand_open)
                delta = newc - base

                if m > 0 and delta > 1e-9 and not _ok_small_positive(delta):
                    continue

                score_cur = (max(delta, 0.0), wait_s)
                if best is None or score_cur < (max(best[0], 0.0), best[3]):
                    best = (delta, tx.taxi_id, pos, wait_s)
        return best

    best = _attempt(candidate_taxi_limit, pos_limit) or _attempt(10**9, 10**9)
    if best is None:
        return False

    _, tid, pos, _ = best
    tx = taxis[tid]
    tx.route_open = tx.route_open[:pos] + [ridx] + tx.route_open[pos:]
    df.at[ridx,'assigned'] = True
    df.at[ridx,'assigned_taxi_id'] = tid
    df.at[ridx,'assigned_depot_id'] = tx.depot_id
    df.at[ridx,'assigned_mode'] = 'insert_open'
    return True

def preempt_if_better(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], ridx_new: int, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, preempt_positions: int = 3
) -> bool:
    ALLOW_SMALL_POS_DELTA = False
    POS_DELTA_CAP_KM = 0.3
    POS_DELTA_FRAC_OF_TRIP = 0.15

    row = df.loc[ridx_new]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']

    def _ok_small_positive(delta: float) -> bool:
        if not ALLOW_SMALL_POS_DELTA:
            return False
        trip_km = haversine_km(row['pickup_lon'], row['pickup_lat'],
                               row['dropoff_lon'], row['dropoff_lat'])
        cap = min(float(POS_DELTA_CAP_KM), float(POS_DELTA_FRAC_OF_TRIP) * max(trip_km, 1e-9))
        return delta <= cap + 1e-9

    cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, candidate_taxi_limit, speed_kmh)

    best: Optional[Tuple[float, str, int, float]] = None
    for tx in cand_taxis:
        m = len(tx.route_open)
        lim = min(preempt_positions, m)
        for pos in range(0, lim+1):
            cand_open = tx.route_open[:pos] + [ridx_new] + tx.route_open[pos:]
            ok, wait_s = route_feasible_from_state(df, tx, [], cand_open, speed_kmh, return_lateness=True)
            if not ok:
                continue
            base = route_cost_km_current(df, tx, [], tx.route_open)
            newc = route_cost_km_current(df, tx, [], cand_open)
            delta = newc - base

            if m > 0 and delta > 1e-9 and not _ok_small_positive(delta):
                continue

            score_cur = (max(delta, 0.0), wait_s)
            if best is None or score_cur < (max(best[0], 0.0), best[3]):
                best = (delta, tx.taxi_id, pos, wait_s)

    if best is None:
        return False

    _, tid, pos, _ = best
    tx = taxis[tid]
    tx.route_open = tx.route_open[:pos] + [ridx_new] + tx.route_open[pos:]
    df.at[ridx_new,'assigned'] = True
    df.at[ridx_new,'assigned_taxi_id'] = tid
    df.at[ridx_new,'assigned_depot_id'] = tx.depot_id
    df.at[ridx_new,'assigned_mode'] = ('preempt_front' if pos == 0 else f'preempt_pos{pos}')
    return True

## 7. Local serch functions on the open paragraph (changes are allowed)

In [ ]:
class MoveStats:
    def __init__(self):
        self.count = {'relocate':0, 'swap':0, 'cross':0}
        self.delta = {'relocate':0.0, 'swap':0.0, 'cross':0.0}


def relocate_best_improvement(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float = 25.0,
                              candidate_taxi_limit: int = 20, pos_limit: int = 6) -> Optional[Tuple[str,float]]:
    import random
    best: Optional[Tuple[float, Tuple]] = None
    tids = list(taxis.keys())
    random.shuffle(tids)
    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma == 0:
            continue
        head_a = list(range(0, min(pos_limit, ma)))
        tail_a = list(range(max(0, ma - pos_limit), ma))
        idx_a = sorted(set(head_a + tail_a))

        for i in idx_a:
            ridx = ta.route_open[i]
            lon_i, lat_i = df.at[ridx,'pickup_lon'], df.at[ridx,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon_i, lat_i, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id
                mb = len(tb.route_open)
                head_b = list(range(0, min(pos_limit, mb)+1))
                tail_b = list(range(max(0, mb - pos_limit), mb+1))
                pos_b = sorted(set(head_b + tail_b))
                for j in pos_b:
                    if a==b and (j==i or j==i+1):
                        continue
                    base = (route_cost_km_current(df, ta, [], ta.route_open)
                            + route_cost_km_current(df, tb, [], tb.route_open))
                    new_a_open = ta.route_open[:i] + ta.route_open[i+1:]
                    new_b_open = tb.route_open[:j] + [ridx] + tb.route_open[j:]
                    if not route_feasible_from_state(df, ta, [], new_a_open, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b_open, speed_kmh):
                        continue
                    newc = route_cost_km_current(df, ta, [], new_a_open) + route_cost_km_current(df, tb, [], new_b_open)
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, ('relocate', a, i, b, j))
    if best:
        delta, (_, a, i, b, j) = best
        ta, tb = taxis[a], taxis[b]
        ridx = ta.route_open.pop(i)
        tb.route_open = tb.route_open[:j] + [ridx] + tb.route_open[j:]
        return ('relocate', delta)
    return None


def swap_best_improvement(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float = 25.0,
                          candidate_taxi_limit: int = 20, pos_limit: int = 6) -> Optional[Tuple[str,float]]:
    best: Optional[Tuple[float, Tuple]] = None
    tids = list(taxis.keys())
    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma == 0:
            continue
        head_a = list(range(0, min(pos_limit, ma)))
        tail_a = list(range(max(0, ma - pos_limit), ma))
        idx_a = sorted(set(head_a + tail_a))

        for i in idx_a:
            ridx = ta.route_open[i]
            lon_i, lat_i = df.at[ridx,'pickup_lon'], df.at[ridx,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon_i, lat_i, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id
                mb = len(tb.route_open)
                if mb == 0 and a != b:
                    continue
                head_b = list(range(0, min(pos_limit, mb)))
                tail_b = list(range(max(0, mb - pos_limit), mb))
                idx_b = sorted(set(head_b + tail_b))
                for j in idx_b:
                    if a==b and i==j:
                        continue
                    base = (route_cost_km_current(df, ta, [], ta.route_open)
                            + route_cost_km_current(df, tb, [], tb.route_open))
                    new_a = ta.route_open.copy(); new_b = tb.route_open.copy()
                    new_a[i], new_b[j] = new_b[j], new_a[i]
                    if not route_feasible_from_state(df, ta, [], new_a, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b, speed_kmh):
                        continue
                    newc = route_cost_km_current(df, ta, [], new_a) + route_cost_km_current(df, tb, [], new_b)
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, ('swap', a, i, b, j))
    if best:
        delta, (_, a, i, b, j) = best
        ta, tb = taxis[a], taxis[b]
        ta.route_open[i], tb.route_open[j] = tb.route_open[j], ta.route_open[i]
        return ('swap', delta)
    return None


def cross_exchange_best_improvement(df: pd.DataFrame, taxis: Dict[str, TaxiLive], k: int = 1, speed_kmh: float = 25.0,
                                    candidate_taxi_limit: int = 20, pos_limit: int = 6) -> Optional[Tuple[str,float]]:
    best: Optional[Tuple[float, Tuple]] = None
    tids = list(taxis.keys())
    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma < k:
            continue
        for i in range(0, max(1, min(pos_limit, ma - k + 1))):
            Ablock = ta.route_open[i:i+k]
            ridx0 = Ablock[0]
            lon0, lat0 = df.at[ridx0,'pickup_lon'], df.at[ridx0,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon0, lat0, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id
                mb = len(tb.route_open)
                if mb < k:
                    continue
                for j in range(0, max(1, min(pos_limit, mb - k + 1))):
                    Bblock = tb.route_open[j:j+k]
                    new_a = ta.route_open[:i] + Bblock + ta.route_open[i+k:]
                    new_b = tb.route_open[:j] + Ablock + tb.route_open[j+k:]
                    if not route_feasible_from_state(df, ta, [], new_a, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b, speed_kmh):
                        continue
                    base = (route_cost_km_current(df, ta, [], ta.route_open)
                            + route_cost_km_current(df, tb, [], tb.route_open))
                    newc = (route_cost_km_current(df, ta, [], new_a)
                            + route_cost_km_current(df, tb, [], new_b))
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, ('cross', a, i, b, j, k))
    if best:
        delta, (_, a, i, b, j, k) = best
        ta, tb = taxis[a], taxis[b]
        Ablock = ta.route_open[i:i+k]
        Bblock = tb.route_open[j:j+k]
        ta.route_open = ta.route_open[:i] + Bblock + ta.route_open[i+k:]
        tb.route_open = tb.route_open[:j] + Ablock + tb.route_open[j+k:]
        return ('cross', delta)
    return None

def cap_open_lists(df: pd.DataFrame, taxis: Dict[str, TaxiLive], m_max_open: int):
    if not m_max_open or m_max_open <= 0:
        return []

    removed_total = []
    for tx in taxis.values():
        m = len(tx.route_open)
        if m <= m_max_open:
            continue

        key_ts = (lambda ridx: df.at[ridx, 'pickup_earliest'] if 'pickup_earliest' in df.columns
                  else df.at[ridx, 'pickup_datetime'])
        idx_sorted_desc = sorted(range(m), key=lambda i: key_ts(tx.route_open[i]), reverse=True)
        remove_idx = set(idx_sorted_desc[: m - m_max_open])

        removed = [tx.route_open[i] for i in sorted(remove_idx)]
        for ridx in removed:
            df.at[ridx, 'assigned'] = False
            df.at[ridx, 'assigned_taxi_id'] = None
            df.at[ridx, 'assigned_depot_id'] = None
            df.at[ridx, 'assigned_mode'] = 'deferred_by_cap'

        tx.route_open = [r for i, r in enumerate(tx.route_open) if i not in remove_idx]
        removed_total.extend(removed)

    return removed_total

## 8.

In [ ]:
def assign_with_ejection(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive],
    ridx_new: int, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 50, pos_limit: int = 12,
    eject_window: int = 2
) -> bool:
    row = df.loc[ridx_new]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']

    cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, candidate_taxi_limit, speed_kmh)

    for ta in cand_taxis:
        m = len(ta.route_open)
        head = list(range(0, min(pos_limit, m) + 1))
        tail = list(range(max(0, m - pos_limit), m + 1))
        positions = sorted(set(head + tail)) if m > pos_limit else list(range(0, m+1))

        for pos in positions:
            cand_open = ta.route_open[:pos] + [ridx_new] + ta.route_open[pos:]
            ok, _late = route_feasible_from_state(df, ta, [], cand_open, speed_kmh, return_lateness=True)
            if ok:
                baseA = route_cost_km_current(df, ta, [], ta.route_open)
                newA  = route_cost_km_current(df, ta, [], cand_open)
                deltaA = newA - baseA
                if deltaA > 1e-9:
                    continue
                ta.route_open = cand_open
                df.at[ridx_new,'assigned'] = True
                df.at[ridx_new,'assigned_taxi_id'] = ta.taxi_id
                df.at[ridx_new,'assigned_depot_id'] = ta.depot_id
                df.at[ridx_new,'assigned_mode'] = 'insert_open_eject0'
                return True

            if m == 0:
                continue
            k_low  = max(0, pos - eject_window)
            k_high = min(m-1, pos + max(0, eject_window-1))
            for k in range(k_low, k_high+1):
                ridx_k = ta.route_open[k]

                open_wo_k = ta.route_open[:k] + ta.route_open[k+1:]
                pos_after = pos if k >= pos else max(0, pos-1)
                open_newA = open_wo_k[:pos_after] + [ridx_new] + open_wo_k[pos_after:]

                okA, _ = route_feasible_from_state(df, ta, [], open_newA, speed_kmh, return_lateness=True)
                if not okA:
                    continue

                baseA = route_cost_km_current(df, ta, [], ta.route_open)
                newA  = route_cost_km_current(df, ta, [], open_newA)
                deltaA = newA - baseA
                if deltaA > 1e-9:
                    continue

                row_k = df.loc[ridx_k]
                k_px_lon, k_px_lat = row_k['pickup_lon'], row_k['pickup_lat']

                target_taxis = [txb for txb in taxis.values() if txb.taxi_id != ta.taxi_id]
                target_taxis.sort(key=lambda txb: _eta_minutes_to_point(txb, k_px_lon, k_px_lat, speed_kmh))
                if candidate_taxi_limit and candidate_taxi_limit > 0:
                    target_taxis = target_taxis[:candidate_taxi_limit]

                best2: Optional[Tuple[float, str, int, float]] = None
                for tb in target_taxis:
                    m2 = len(tb.route_open)
                    for pos2 in range(0, m2 + 1):
                        cand_open2 = tb.route_open[:pos2] + [ridx_k] + tb.route_open[pos2:]
                        okB, lateB = route_feasible_from_state(df, tb, [], cand_open2, speed_kmh, return_lateness=True)
                        if not okB:
                            continue

                        baseB = route_cost_km_current(df, tb, [], tb.route_open)
                        newcB = route_cost_km_current(df, tb, [], cand_open2)
                        deltaB = newcB - baseB

                        if deltaB > 1e-9:
                            continue

                        if best2 is None or (deltaB, lateB) < (best2[0], best2[3]):
                            best2 = (deltaB, tb.taxi_id, pos2, lateB)

                if best2 is not None:
                    _, tbid, pos2, _ = best2
                    tb = taxis[tbid]
                    tb.route_open = tb.route_open[:pos2] + [ridx_k] + tb.route_open[pos2:]
                    df.at[ridx_k,'assigned'] = True
                    df.at[ridx_k,'assigned_taxi_id'] = tb.taxi_id
                    df.at[ridx_k,'assigned_depot_id'] = tb.depot_id
                    df.at[ridx_k,'assigned_mode'] = 'reinsert_after_eject'

                    ta.route_open = open_newA
                    df.at[ridx_new,'assigned'] = True
                    df.at[ridx_new,'assigned_taxi_id'] = ta.taxi_id
                    df.at[ridx_new,'assigned_depot_id'] = ta.depot_id
                    df.at[ridx_new,'assigned_mode'] = f'insert_open_eject1(k={ridx_k})'
                    return True
    return False

## 9. Pull new request every tick

In [ ]:

def pull_new_requests(df: pd.DataFrame, last_now: pd.Timestamp, now: pd.Timestamp) -> List[int]:
    """
    Retry ALL unassigned & unexpired requests.
    """
    base_mask = (
        (df["assigned"] == False)
        & (df["assigned_mode"] != "rejected")
        & (df["pickup_earliest"] <= now)
    )

    has_latest = "pickup_latest" in df.columns
    if has_latest:
        not_expired = (df["pickup_latest"].isna()) | (df["pickup_latest"] >= now)
    else:
        not_expired = pd.Series(True, index=df.index)

    mask_pending = base_mask & not_expired
    if not mask_pending.any():
        return []

    if has_latest:
        far_future = now + pd.Timedelta(days=3650)
        order = df.loc[mask_pending, ["pickup_latest", "pickup_earliest"]].copy()
        order["pickup_latest"] = pd.to_datetime(order["pickup_latest"]).fillna(far_future)
        order["pickup_earliest"] = pd.to_datetime(order["pickup_earliest"])
        idx = order.sort_values(["pickup_latest", "pickup_earliest"]).index.tolist()
    else:
        order = df.loc[mask_pending, ["pickup_earliest"]].copy()
        idx = order.sort_values(["pickup_earliest"]).index.tolist()

    return idx

## 10. The act of returning to the depot when free

In [ ]:
def reposition_idle_taxis_to_depot(taxis: Dict[str, TaxiLive], now: pd.Timestamp,
                                   speed_kmh: float, eps_km: float = 0.05) -> None:
    """
    For each taxi that is free (no longer route_freeze/open), move it to the depot (origin)
    with speed_kmh. Update current location & time to 'now' (no virtual request).
    eps_km: threshold considered to be at the depot.
    """
    if speed_kmh <= 0:
        return

    for tx in taxis.values():
        is_freeze_done = (tx.freeze_ptr == len(tx.route_freeze))
        if not (is_freeze_done and len(tx.route_open) == 0):
            continue

        if tx.cur_time >= now:
            continue

        allow_min = (now - tx.cur_time).total_seconds() / 60.0
        if allow_min <= 0:
            continue

        dist_km = haversine_km(tx.cur_lon, tx.cur_lat, tx.origin_lon, tx.origin_lat)

        if dist_km <= eps_km:
            tx.cur_lon, tx.cur_lat = tx.origin_lon, tx.origin_lat
            tx.cur_time = now
            continue

        move_km = speed_kmh * (allow_min / 60.0)

        moved_km = min(move_km, dist_km)
        tx.reposition_km += moved_km

        if move_km >= dist_km - 1e-9:
            travel_min = 60.0 * dist_km / speed_kmh
            tx.cur_lon, tx.cur_lat = tx.origin_lon, tx.origin_lat
            tx.cur_time = now
        else:
            ratio = move_km / max(dist_km, 1e-12)
            tx.cur_lon = tx.cur_lon + ratio * (tx.origin_lon - tx.cur_lon)
            tx.cur_lat = tx.cur_lat + ratio * (tx.origin_lat - tx.cur_lat)
            tx.cur_time = tx.cur_time + pd.to_timedelta(allow_min, unit='m')

## 11. Main loop

In [ ]:

def run_dynamic_no_horizon(df_out: pd.DataFrame, fleet: Fleet,
                           tick_s: int = 30, freeze_window_s: int = 20,
                           speed_kmh: float = 25.0, max_ops_per_tick: int = 50,
                           candidate_taxi_limit: int = 20, pos_limit: int = 6,
                           m_max_open: int = 8, log_every: int = 60,
                           use_ejection: bool = True, eject_window: int = 2) -> Tuple[pd.DataFrame, Dict, Dict[str, TaxiLive]]:
    df = df_out.copy()

    for c in ['sim_pickup_datetime', 'sim_dropoff_datetime']:
        if c not in df.columns:
            df[c] = pd.NaT
        else:
            df[c] = pd.to_datetime(df[c], errors='coerce')

    if 'assigned' not in df.columns:
        df['assigned'] = False
    else:
        df['assigned'] = df['assigned'].fillna(False)

    for c, default in [('assigned_taxi_id', None), ('assigned_depot_id', None)]:
        if c not in df.columns:
            df[c] = pd.Series([default]*len(df), dtype='object')
        else:
            df[c] = df[c].astype('object')

    if 'assigned_mode' not in df.columns:
        df['assigned_mode'] = ''
    else:
        df['assigned_mode'] = df['assigned_mode'].fillna('').astype('object')

    taxis = build_taxis_live(df, fleet)

    t_min = pd.to_datetime(df['pickup_earliest'] if 'pickup_earliest' in df.columns else df['pickup_datetime']).min()
    t_max = pd.to_datetime(df['pickup_latest'] if 'pickup_latest' in df.columns else df['pickup_datetime']).max()

    now = t_min
    end = t_max + pd.Timedelta(seconds=tick_s)

    stats = {'moves': MoveStats()}

    last_now = now - pd.Timedelta(seconds=tick_s)
    while now <= end:
        _tick_t0 = time.perf_counter()

        update_freeze_open(df, taxis, now, freeze_window_s=freeze_window_s)
        advance_state_after_freeze(df, taxis, speed_kmh)
        cap_open_lists(df, taxis, m_max_open)

        reposition_idle_taxis_to_depot(taxis, now, speed_kmh)

        new_ids = pull_new_requests(df, last_now, now)
        rej_this_tick = 0

        for ridx in new_ids:
            ok = preempt_if_better(df, taxis, ridx, speed_kmh=speed_kmh,
                                   candidate_taxi_limit=candidate_taxi_limit,
                                   preempt_positions=min(3, pos_limit),
                                   ) or try_insert_best(df, taxis, ridx, speed_kmh=speed_kmh,
                                                        candidate_taxi_limit=candidate_taxi_limit,
                                                        pos_limit=pos_limit)

            if (not ok) and use_ejection:
                ok = assign_with_ejection(df, taxis, ridx, speed_kmh=speed_kmh,
                                          candidate_taxi_limit=max(40, candidate_taxi_limit),
                                          pos_limit=max(12, pos_limit),
                                          eject_window=eject_window)

            if not ok:
                px_l = df.at[ridx, 'pickup_latest'] if 'pickup_latest' in df.columns else df.at[ridx, 'pickup_datetime']
                if pd.notna(px_l) and now >= px_l:
                    df.at[ridx,'assigned'] = False
                    df.at[ridx,'assigned_mode'] = 'rejected'
                    rej_this_tick += 1

        ops = 0
        num_new = len(new_ids)
        while ops < max_ops_per_tick:
            cand = [
                relocate_best_improvement(df, taxis, speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit),
                swap_best_improvement(df, taxis, speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit),
                cross_exchange_best_improvement(df, taxis, k=1, speed_kmh=speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit)
            ]
            cand = [c for c in cand if c is not None]
            if not cand:
                break
            name, delta = min(cand, key=lambda x: x[1])
            stats['moves'].count[name] += 1
            stats['moves'].delta[name] += delta
            ops += 1

        tick_idx = int((now - t_min).total_seconds() // tick_s)
        if log_every and (tick_idx % log_every == 0):
            _tick_dt = time.perf_counter() - _tick_t0
            freeze_cnt = sum(len(tx.route_freeze) for tx in taxis.values())
            open_cnt = sum(len(tx.route_open) for tx in taxis.values())
            print(f"[tick {now}] new={num_new}, rej={rej_this_tick}, ops={ops}, freeze={freeze_cnt}, open={open_cnt}, tick_time={_tick_dt:.3f}s")

        last_now = now
        now += pd.Timedelta(seconds=tick_s)

    for tx in taxis.values():
        for ridx in tx.route_freeze + tx.route_open:
            df.at[ridx, 'assigned'] = True
            df.at[ridx, 'assigned_taxi_id'] = tx.taxi_id
            df.at[ridx, 'assigned_depot_id'] = tx.depot_id

    materialize_sim_times(df, taxis, speed_kmh)

    deadline = end
    mask_to_close = (
        (df['assigned'] == False) &
        (pd.to_datetime(df['pickup_latest'] if 'pickup_latest' in df.columns else df['pickup_datetime']) <= deadline)
    )
    df.loc[mask_to_close, 'assigned_mode'] = 'rejected'

    used_taxis = set(df.loc[df['assigned'] == True, 'assigned_taxi_id'].dropna().astype(str).tolist())
    total_served = int((df['assigned'] == True).sum())
    total_rejected = int((df.get('assigned_mode','') == 'rejected').sum())

    total_km = 0.0
    dead_km = 0.0
    per_taxi_km = {}

    for tid in used_taxis:
        tx = taxis[tid]
        seq = tx.route_freeze + tx.route_open

        km_total = route_cost_km_from_origin(df, seq, tx.origin_lon, tx.origin_lat)
        km_dead = deadhead_km_from_origin(df, seq, tx.origin_lon, tx.origin_lat)

        total_km += km_total
        dead_km += km_dead
        per_taxi_km[tid] = {"total_km": km_total, "deadhead_km": km_dead}

    if per_taxi_km:
        total_vals = np.array([v["total_km"] for v in per_taxi_km.values()], dtype=float)
        dead_vals  = np.array([v["deadhead_km"] for v in per_taxi_km.values()], dtype=float)

        taxi_total_km_max = float(total_vals.max())
        taxi_total_km_min = float(total_vals.min())
        taxi_total_km_avg = float(total_vals.mean())

        taxi_deadhead_km_max = float(dead_vals.max())
        taxi_deadhead_km_min = float(dead_vals.min())
        taxi_deadhead_km_avg = float(dead_vals.mean())
    else:
        taxi_total_km_max = taxi_total_km_min = taxi_total_km_avg = 0.0
        taxi_deadhead_km_max = taxi_deadhead_km_min = taxi_deadhead_km_avg = 0.0

    reposition_km = float(sum(getattr(taxis[tid], "reposition_km", 0.0) for tid in used_taxis))

    df, profit_by_taxi_paper, system_profit_paper = compute_profit_paper_style(
        df, taxis, speed_kmh=speed_kmh
    )
    system_fare_paper = float(sum(v["total_fare_paper"] for v in profit_by_taxi_paper.values()))
    system_cost_drive_paper = float(sum(v["total_cost_drive_paper"] for v in profit_by_taxi_paper.values()))
    system_cost_wait_paper  = float(sum(v["total_cost_wait_paper"] for v in profit_by_taxi_paper.values()))

    if "km_total_paper" in df.columns:
        served_mask = (df["assigned"] == True) & df["sim_pickup_datetime"].notna()
        if served_mask.any():
            req_km = df.loc[served_mask, "km_total_paper"].to_numpy(dtype=float)
            req_km = req_km[~np.isnan(req_km)]
            if len(req_km) > 0:
                request_km_max = float(req_km.max())
                request_km_min = float(req_km.min())
            else:
                request_km_max = request_km_min = 0.0
        else:
            request_km_max = request_km_min = 0.0
    else:
        request_km_max = request_km_min = 0.0

    report = {
        "num_taxis_used": len(used_taxis),
        "served": int(total_served),
        "rejected": int(total_rejected),

        "total_km": float(total_km),
        "deadhead_km": float(dead_km),
        "reposition_km": float(reposition_km),

        "system_fare": system_fare_paper,
        "system_cost_drive": system_cost_drive_paper,
        "system_cost_wait": system_cost_wait_paper,
        "system_profit": float(system_profit_paper),

        "taxi_total_km_max": taxi_total_km_max,
        "taxi_total_km_min": taxi_total_km_min,
        "taxi_total_km_avg": taxi_total_km_avg,
        "taxi_deadhead_km_max": taxi_deadhead_km_max,
        "taxi_deadhead_km_min": taxi_deadhead_km_min,
        "taxi_deadhead_km_avg": taxi_deadhead_km_avg,
        "request_km_max": request_km_max,
        "request_km_min": request_km_min,

        "move_counts": stats["moves"].count,
        "move_effects_km": stats["moves"].delta,
    }
    return df, report, taxis

# Run dynamic with 10 data files from the drive and export the results

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging, os, re

try:
    from google.colab import drive
    else:
        print("Drive already mounted.")
except Exception as e:
    print("Note:", e)

DATA_PATH = Path(r"d:/DTRP-DP/Routing/Input")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

RESULTS_DIR = Path(r"d:/DTRP-DP/Routing/Output/dynamic_v2_outputs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV  = RESULTS_DIR / "reports_for_last_monthver1.csv"

SUMMARY_COLUMNS = [
    "File",
    "datetime",
    "num taxi used",
    "total km",
    "deadhead km",
    "served",
    "rejected",
    "relocate", "swap", "cross",
]

def _flatten_report(rpt: dict) -> dict:
    """Break the child dicts (move_counts/move_effects_km) into flat columns."""
    out = dict(rpt or {})

    mc = out.pop("move_counts", None) or {}
    for k, v in mc.items():
        out[f"move_counts_{k}"] = int(v)

    me = out.pop("move_effects_km", None) or {}
    for k, v in me.items():
        out[f"move_effects_km_{k}"] = float(v)

    return out


PREFERRED_ORDER = [
    "File",
    "datetime",
    "num_taxis_used",
    "served",
    "rejected",
    "total_km",
    "deadhead_km",
    "reposition_km",

    "system_fare",
    "system_cost_drive",
    "system_cost_wait",
    "system_profit",

    "taxi_total_km_max",
    "taxi_total_km_min",
    "taxi_total_km_avg",
    "taxi_deadhead_km_max",
    "taxi_deadhead_km_min",
    "taxi_deadhead_km_avg",
    "request_km_max",
    "request_km_min",

    "move_counts_relocate",
    "move_counts_swap",
    "move_counts_cross",
    "move_effects_km_relocate",
    "move_effects_km_swap",
    "move_effects_km_cross",
]

def save_report_to_csv(report: dict, file_rel_path: str, date_str: str, csv_path: Path):
    """
    Write 1 line of report into CSV:
    - automatically add new columns if the report has a new key
    - keep the order according to PREFERRED_ORDER, strange columns are pushed to the back
    """
    flat = _flatten_report(report)
    row = {"File": file_rel_path, "datetime": date_str, **flat}

    if csv_path.exists() and csv_path.stat().st_size > 0:
        old = pd.read_csv(csv_path)
        df_out = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    else:
        df_out = pd.DataFrame([row])

    cols = list(df_out.columns)
    ordered = [c for c in PREFERRED_ORDER if c in cols]
    extras = [c for c in cols if c not in ordered]
    df_out = df_out[ordered + extras]

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
    return csv_path

def _format_move_cell(name: str, rpt: dict) -> str:
    c = (rpt.get('move_counts', {}) or {}).get(name, 0) or 0
    d = (rpt.get('move_effects_km', {}) or {}).get(name, 0.0) or 0.0
    return f"{c} times, cumulative Δkm = {d:.2f}"

def _pick_day_value(df: pd.DataFrame) -> str:
    if 'pickup_datetime' in df.columns and df['pickup_datetime'].notna().any():
        return pd.to_datetime(df['pickup_datetime']).dt.date.min().isoformat()
    if 'pickup_earliest' in df.columns and df['pickup_earliest'].notna().any():
        return pd.to_datetime(df['pickup_earliest']).dt.date.min().isoformat()
    return ""

def scale_taxis_per_depot(df: pd.DataFrame, factor: float = 2.5, rounding: str = "ceil") -> pd.DataFrame:
    df = df.copy()
    if 'taxis_per_depot' not in df.columns:
        df['taxis_per_depot'] = 1
    x = pd.to_numeric(df['taxis_per_depot'], errors='coerce').fillna(0)
    if rounding == "ceil":
        df['taxis_per_depot'] = np.ceil(x * factor).astype(int)
    elif rounding == "round":
        df['taxis_per_depot'] = np.rint(x * factor).astype(int)
    else:
        df['taxis_per_depot'] = np.floor(x * factor).astype(int)
    df['taxis_per_depot'] = df['taxis_per_depot'].clip(lower=0)
    return df

def _read_csv_simple(file_path: Path) -> pd.DataFrame:
    """Read minimalist CSV, force datetime for pickup/dropoff if available."""
    df = pd.read_csv(file_path, low_memory=False)
    for c in ("pickup_datetime", "dropoff_datetime"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

def load_one_file(file_rel_path: str):
    """
    Correctly read a CSV file according to the relative path from DATA_PATH.
    Returns DataFrame.
    """
    file_path = DATA_PATH / file_rel_path
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path, low_memory=False)

    for c in ("pickup_datetime", "dropoff_datetime"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    print(f"Loaded: {file_path.name}")
    print(f" - Shape: {df.shape}")
    if "pickup_datetime" in df.columns and not df["pickup_datetime"].isna().all():
        print(f"- Time interval: {df['pickup_datetime'].min()} → {df['pickup_datetime'].max()}")

    return df

def _read_input(file_rel_path: str) -> pd.DataFrame:
    """Prefer to use old-style load_one_file if available; fallback reads directly."""
    if 'load_one_file' in globals():
        return load_one_file(file_rel_path)
    return _read_csv_simple(DATA_PATH / file_rel_path)

def _extract_day_from_filename(p: Path) -> str | None:
    m = re.search(r'(20\d{2})[-_]?(\d{2})[-_]?(\d{2})', p.name)
    if not m:
        return None
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

def _name_or_date_to_key(x: str) -> str | None:
    """
    Get 'requests_2016-05-14.csv' or '2016-05-14' and return '2016-05-14'.
    """
    m = re.search(r'(20\d{2})[-_]?(\d{2})[-_]?(\d{2})', x)
    if not m:
        return None
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

def run_dynamic_one_and_append_csv(
    file_rel_path: str,
    taxi_factor: float = 1.0,
    tick_s: int = 60,
    freeze_s: int = 30,
    speed_kmh: float = 25.0,
    max_ops_per_tick: int = 80,
    pos_limit: int = 16,
    m_max_open: int = 14,
    log_every: int = 60,
    csv_path: Path = RESULT_CSV
):
    file_abs = DATA_PATH / file_rel_path
    logging.info(f"Loading file: {file_abs}")
    day_df = _read_input(file_rel_path)

    if taxi_factor and taxi_factor != 1.0:
        day_df = scale_taxis_per_depot(day_df, factor=taxi_factor, rounding="ceil")

    req = day_df.copy()
    for c in ['assigned','assigned_taxi_id','assigned_depot_id','assigned_mode']:
        if c not in req.columns:
            req[c] = None
    req['assigned'] = False
    req['assigned_taxi_id'] = None
    req['assigned_depot_id'] = None
    req['assigned_mode'] = ""

    req = normalize_requests_df(req)
    req = ensure_pickup_windows(req, early_s=0, late_s=360)
    req = req.sort_values('pickup_earliest').reset_index(drop=True)

    fleet_sim = Fleet.from_depots(req)

    cand_limit = 153

    df_dyn, rpt, taxis = run_dynamic_no_horizon(
        df_out=req,
        fleet=fleet_sim,
        tick_s=tick_s,
        freeze_window_s=freeze_s,
        speed_kmh=speed_kmh,
        max_ops_per_tick=max_ops_per_tick,
        candidate_taxi_limit=cand_limit,
        pos_limit=pos_limit,
        m_max_open=m_max_open,
        log_every=log_every,
        use_ejection=True,
        eject_window=2
    )

    date_str = _pick_day_value(req)
    save_report_to_csv(
        report=rpt,
        file_rel_path=file_rel_path,
        date_str=date_str,
        csv_path=csv_path
    )

    logging.info(f"Wrote full report to: {csv_path}")
    return {"File": file_rel_path, "datetime": date_str, **_flatten_report(rpt)}, df_dyn, rpt, taxis


def run_batch_files(
    input_dir_rel: str = "last_month_daily/last_month_daily_eff_3",
    pattern: str = "requests_*.csv",
    limit: int | None = 10,
    taxi_factor: float = 1.0,
    resume: bool = True,
    start_from: str | None = None
):
    dir_path = (DATA_PATH / input_dir_rel).resolve()
    logging.info(f"Scanning dir: {dir_path} | pattern: {pattern}")
    if not dir_path.exists():
        logging.error(f"Input dir not found: {dir_path}")
        return

    files_csv = sorted(list(dir_path.glob(pattern)))
    files_gz = []
    if pattern.endswith(".csv"):
        files_gz = sorted(list(dir_path.glob(pattern + ".gz")))
    files = sorted(files_csv + files_gz)

    logging.info(f"Found {len(files)} files. Sample: {[p.name for p in files[:3]]}")
    if not files:
        logging.warning("No files matching pattern were found.")
        return

    if start_from:
        start_key = _name_or_date_to_key(start_from)
        if start_key:
            idx = None
            for i, p in enumerate(files):
                if p.name == start_from:
                    idx = i
                    break
            if idx is not None:
                files = files[idx:]
            else:
                files = [p for p in files if (_extract_day_from_filename(p) or "") >= start_key]
        else:
            logging.warning(f"start_from='{start_from}' does not recognize dates, ignore this parameter.")

    if isinstance(limit, int) and limit is not None and limit > 0:
        files = files[:limit]

    if RESULT_CSV.exists() and not resume:
        RESULT_CSV.unlink()

    processed_files = set()
    processed_days  = set()
    if RESULT_CSV.exists():
        try:
            prev = pd.read_csv(RESULT_CSV, dtype=str)
            if 'File' in prev.columns:
                processed_files = set(prev['File'].dropna().astype(str))
            if 'datetime' in prev.columns:
                processed_days = set(prev['datetime'].dropna().astype(str))
            elif 'Days' in prev.columns:
                processed_days = set(prev['Days'].dropna().astype(str))
        except Exception as e:
            logging.warning(f"Unable to read old CSV for resume: {e}")

    for i, p in enumerate(files, 1):
        rel_to_dir = p.relative_to(dir_path)
        rel = str((Path(input_dir_rel) / rel_to_dir).as_posix())

        if rel in processed_files:
            logging.info(f"[{i}/{len(files)}] SKIP (File): {rel}")
            continue

        day_in_name = _extract_day_from_filename(p)
        if day_in_name and day_in_name in processed_days:
            logging.info(f"[{i}/{len(files)}] SKIP (Day): {rel} ({day_in_name})")
            continue

        logging.info(f"[{i}/{len(files)}] RUN: {rel}")
        try:
            run_dynamic_one_and_append_csv(
                rel,
                taxi_factor=taxi_factor,
                tick_s=60,
                freeze_s=30,
                speed_kmh=25.0,
                max_ops_per_tick=50,
                pos_limit=16,
                m_max_open=25,
                log_every=60,
                csv_path=RESULT_CSV
            )
        except Exception as e:
            logging.warning(f"Ignore {rel} due to error: {e}")

from datetime import timedelta

INPUT_DIR  = "last_month_daily/last_month_daily_eff_3"
PATTERN    = "requests_*.csv"
LIMIT_EACH = 10

run_batch_files(
    input_dir_rel=INPUT_DIR,
    pattern=PATTERN,
    start_from="2016-06-14",
    limit=LIMIT_EACH,
    taxi_factor=1.0,
    resume=True
)

def _pick_datetime_column(df: pd.DataFrame) -> str:
    if 'datetime' in df.columns:
        return 'datetime'
    if 'Days' in df.columns:
        return 'Days'
    raise ValueError("RESULT_CSV does not have a 'datetime' or 'Dates' column.")

def _last_processed_day_for_dir(csv_path: Path, input_dir_rel: str) -> str | None:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return None
    prev = pd.read_csv(csv_path, dtype=str)
    col_day = _pick_datetime_column(prev)
    if 'File' not in prev.columns:
        raise ValueError("RESULT_CSV is missing column 'File'.")
    mask = prev['File'].astype(str).str.startswith(f"{input_dir_rel}/")
    prev_dir = prev.loc[mask].copy()
    if prev_dir.empty:
        return None
    dt = pd.to_datetime(prev_dir[col_day], errors='coerce')
    if dt.notna().any():
        return dt.max().date().isoformat()
    return None

_last = _last_processed_day_for_dir(RESULT_CSV, INPUT_DIR)
if _last is None:
    base = pd.to_datetime("2016-05-14").date()
    next_start = (base + timedelta(days=LIMIT_EACH)).isoformat()
else:
    next_start = (pd.to_datetime(_last).date() + timedelta(days=1)).isoformat()

run_batch_files(
    input_dir_rel=INPUT_DIR,
    pattern=PATTERN,
    start_from=next_start,
    limit=LIMIT_EACH,
    taxi_factor=1.0,
    resume=True
)

print("✅ Done. Check out the summary at:", RESULT_CSV)


Drive already mounted.


/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


Đã nạp: requests_2016-06-14.csv
 - Shape: (7470, 25)
 - Khoảng thời gian: 2016-06-14 00:00:11 → 2016-06-14 23:59:57
[tick 2016-06-14 00:00:11] new=1, rej=0, ops=0, freeze=0, open=1, tick_time=0.296s
[tick 2016-06-14 01:00:11] new=6, rej=0, ops=0, freeze=159, open=3, tick_time=1.242s
[tick 2016-06-14 02:00:11] new=4, rej=0, ops=0, freeze=239, open=2, tick_time=0.396s
[tick 2016-06-14 03:00:11] new=2, rej=0, ops=0, freeze=279, open=1, tick_time=0.272s
[tick 2016-06-14 04:00:11] new=0, rej=0, ops=0, freeze=312, open=0, tick_time=0.003s
[tick 2016-06-14 05:00:11] new=0, rej=0, ops=0, freeze=336, open=0, tick_time=0.003s
[tick 2016-06-14 06:00:11] new=9, rej=0, ops=0, freeze=377, open=3, tick_time=1.010s
[tick 2016-06-14 07:00:11] new=7, rej=0, ops=0, freeze=564, open=5, tick_time=0.745s
[tick 2016-06-14 08:00:11] new=14, rej=0, ops=0, freeze=823, open=4, tick_time=1.572s
[tick 2016-06-14 09:00:11] new=9, rej=0, ops=0, freeze=1140, open=7, tick_time=1.012s
[tick 2016-06-14 10:00:11] new=13,

/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


Đã nạp: requests_2016-06-16.csv
 - Shape: (8289, 25)
 - Khoảng thời gian: 2016-06-16 00:00:02 → 2016-06-16 23:59:43
[tick 2016-06-16 00:00:02] new=1, rej=0, ops=0, freeze=0, open=1, tick_time=0.152s
[tick 2016-06-16 01:00:02] new=5, rej=0, ops=1, freeze=203, open=5, tick_time=1.284s
[tick 2016-06-16 02:00:02] new=1, rej=0, ops=0, freeze=329, open=0, tick_time=0.112s
[tick 2016-06-16 03:00:02] new=0, rej=0, ops=0, freeze=406, open=0, tick_time=0.005s
[tick 2016-06-16 04:00:02] new=8, rej=0, ops=1, freeze=454, open=2, tick_time=1.699s
[tick 2016-06-16 05:00:02] new=1, rej=0, ops=0, freeze=501, open=0, tick_time=0.112s
[tick 2016-06-16 06:00:02] new=4, rej=0, ops=0, freeze=544, open=2, tick_time=0.438s
[tick 2016-06-16 07:00:02] new=10, rej=0, ops=0, freeze=773, open=6, tick_time=1.098s
[tick 2016-06-16 08:00:02] new=16, rej=0, ops=0, freeze=1125, open=5, tick_time=1.738s
[tick 2016-06-16 09:00:02] new=10, rej=0, ops=0, freeze=1483, open=2, tick_time=1.089s
[tick 2016-06-16 10:00:02] new=

...

/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


Đã nạp: requests_2016-06-23.csv
 - Shape: (7875, 25)
 - Khoảng thời gian: 2016-06-23 00:00:36 → 2016-06-23 23:59:59
[tick 2016-06-23 00:00:36] new=1, rej=0, ops=0, freeze=0, open=1, tick_time=0.131s
[tick 2016-06-23 01:00:36] new=4, rej=0, ops=0, freeze=219, open=3, tick_time=0.429s
[tick 2016-06-23 02:00:36] new=7, rej=0, ops=0, freeze=333, open=4, tick_time=0.742s
[tick 2016-06-23 03:00:36] new=4, rej=0, ops=0, freeze=405, open=2, tick_time=0.438s
[tick 2016-06-23 04:00:36] new=0, rej=0, ops=0, freeze=452, open=0, tick_time=0.004s
[tick 2016-06-23 05:00:36] new=0, rej=0, ops=0, freeze=487, open=0, tick_time=0.004s
[tick 2016-06-23 06:00:36] new=4, rej=1, ops=0, freeze=543, open=1, tick_time=0.408s
[tick 2016-06-23 07:00:36] new=12, rej=0, ops=0, freeze=704, open=6, tick_time=1.344s
[tick 2016-06-23 08:00:36] new=10, rej=0, ops=0, freeze=986, open=8, tick_time=1.305s
[tick 2016-06-23 09:00:36] new=6, rej=0, ops=0, freeze=1350, open=1, tick_time=1.113s
[tick 2016-06-23 10:00:36] new=13

/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-916688654.py:162: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


Đã nạp: requests_2016-06-24.csv
 - Shape: (7946, 25)
 - Khoảng thời gian: 2016-06-24 00:00:01 → 2016-06-24 23:59:57
[tick 2016-06-24 00:00:01] new=1, rej=0, ops=0, freeze=0, open=0, tick_time=0.165s
[tick 2016-06-24 01:00:01] new=9, rej=0, ops=1, freeze=298, open=5, tick_time=1.421s
[tick 2016-06-24 02:00:01] new=6, rej=0, ops=0, freeze=511, open=3, tick_time=0.613s
[tick 2016-06-24 03:00:01] new=3, rej=0, ops=1, freeze=634, open=3, tick_time=0.789s
[tick 2016-06-24 04:00:01] new=3, rej=0, ops=0, freeze=713, open=1, tick_time=0.336s
[tick 2016-06-24 05:00:01] new=3, rej=0, ops=0, freeze=777, open=1, tick_time=0.370s
[tick 2016-06-24 06:00:01] new=2, rej=0, ops=0, freeze=854, open=0, tick_time=0.213s


KeyboardInterrupt: 